# 4-channel 3D U-Net training (seed=42) on MAMA-MIA

**Probe goal:** test if 4-channel input `[pre, post1, post2, subtraction]` + phase augmentation closes any of the gap to the MAMA-MIA paper baseline (0.762 nnU-Net, 5-fold CV).

**Comparison rule:** Î” vs 2ch baseline s=42 = 0.6791.
- Î” â‰¥ +0.020 â†’ real win, justifies redoing FADC variants at 4ch
- +0.005 â‰¤ Î” < +0.020 â†’ marginal, future-work
- Î” < +0.005 â†’ no win, file as negative ablation

**Setup:**
- Cache: `vbk1999/4ch-mama-mia-cache` (must be attached as Kaggle dataset)
- Code: cloned from `https://github.com/Vemuri-BK/FADC-3D.git` (notebooks are gitignored â€” only .py files come down)
- Seed: 42 (controlled, `cudnn.deterministic=True`)
- Phase augmentation: ON (p=0.5)
- 100 epochs, batch=2, 5-epoch warmup, cosine decay to 1e-6

## 1. Config

In [ ]:
import os

# === EDIT IF NEEDED ===
SEED                = 42
EPOCHS              = 100
BATCH_SIZE          = 2
LR                  = 1.0e-4
PHASE_AUG           = 1        # 1=on, 0=off
P_PHASE_AUG         = 0.5

# Kaggle's actual mount path (with old-style datasets/<owner>/ prefix).
# The zip upload produced double-nested folders, so the real .npz files are at:
#   <KAGGLE_MOUNT>/train/train/*.npz  and  <KAGGLE_MOUNT>/val/val/*.npz
# Cell 6 below symlinks these into a flat layout at PREPROCESSED_CACHE.
KAGGLE_MOUNT        = "/kaggle/input/datasets/vbk1999/4ch-mama-mia-cache"
PREPROCESSED_CACHE  = "/kaggle/working/4ch-cache"
SPLIT_CSV           = f"{PREPROCESSED_CACHE}/train_test_splits.csv"

# Where to save outputs inside the Kaggle session
OUTPUT_DIR          = f"/kaggle/working/outputs/4ch_unet3d_100ep_s{SEED}"

# Patch size: same as 2ch baseline for apples-to-apples
PATCH_X, PATCH_Y, PATCH_Z = 128, 128, 64

# Code location (where we clone the repo)
REPO_DIR            = "/kaggle/working/FADC-3D"
REPO_URL            = "https://github.com/Vemuri-BK/FADC-3D.git"

os.makedirs("/kaggle/working/outputs", exist_ok=True)

print(f"SEED              : {SEED}")
print(f"EPOCHS            : {EPOCHS}")
print(f"PHASE_AUG         : {PHASE_AUG} (p={P_PHASE_AUG})")
print(f"KAGGLE_MOUNT      : {KAGGLE_MOUNT}")
print(f"PREPROCESSED_CACHE: {PREPROCESSED_CACHE}  (will be created by cell 6)")
print(f"OUTPUT_DIR        : {OUTPUT_DIR}")


## 2. Dependencies + repo clone

Installs MONAI and clones the FADC-3D repo (which contains the 4ch dataset module and training script). Notebooks are gitignored in the repo so the clone only brings down .py files â€” this notebook stays in Kaggle.

In [ ]:
!pip install -q monai tqdm pyyaml 2>&1 | tail -3

In [ ]:
import subprocess

# The 4ch files (data/mama_mia_dataset_4ch.py, training/train_centralized_4ch.py,
# configs/config_4ch.yaml) live on the experiment/4ch-probe branch, NOT main.
BRANCH = "experiment/4ch-probe"

if not os.path.exists(REPO_DIR):
    subprocess.run(
        ["git", "clone", "-b", BRANCH, "--depth", "1", REPO_URL, REPO_DIR],
        check=True,
    )
    print(f"Cloned repo to {REPO_DIR}  (branch: {BRANCH})")
else:
    print(f"Repo already present at {REPO_DIR}")
    head = subprocess.check_output(
        ["git", "-C", REPO_DIR, "rev-parse", "--abbrev-ref", "HEAD"]
    ).decode().strip()
    if head != BRANCH:
        print(f"  Existing clone is on '{head}', switching to '{BRANCH}'...")
        subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "checkout", BRANCH], check=True)

# Verify the 4ch files are present
required = [
    f"{REPO_DIR}/data/mama_mia_dataset_4ch.py",
    f"{REPO_DIR}/training/train_centralized_4ch.py",
    f"{REPO_DIR}/configs/config_4ch.yaml",
    f"{REPO_DIR}/models/unet_3d.py",
]
for path in required:
    assert os.path.exists(path), f"Missing: {path} -- push the 4ch files to GitHub first!"
print("All 4ch files present in the repo.")


## 2b. Flatten the cache layout via symlinks

The .npz files were uploaded as zips and Kaggle auto-extracted them, leaving a double-nested layout: `train/train/*.npz` instead of `train/*.npz`. This cell creates symlinks in `/kaggle/working/4ch-cache/` so the rest of the notebook sees the clean flat layout the dataset module expects. Symlinks are instant and use zero extra disk.

In [ ]:
import os, shutil
from pathlib import Path

os.makedirs(PREPROCESSED_CACHE, exist_ok=True)

assert os.path.exists(KAGGLE_MOUNT), (
    f"Kaggle mount not found: {KAGGLE_MOUNT}\n"
    f"Attach the 'vbk1999/4ch-mama-mia-cache' dataset in the notebook sidebar."
)

for split in ("train", "val"):
    link   = f"{PREPROCESSED_CACHE}/{split}"
    target = f"{KAGGLE_MOUNT}/{split}/{split}"   # double-nested source
    assert os.path.exists(target), f"Source not found: {target}"
    if os.path.islink(link):
        os.unlink(link)
    elif os.path.exists(link):
        shutil.rmtree(link)
    os.symlink(target, link)

# Copy train_test_splits.csv (small, ~50KB)
splits_src = f"{KAGGLE_MOUNT}/train_test_splits.csv"
if os.path.exists(splits_src):
    shutil.copy(splits_src, SPLIT_CSV)

# Verify
for split in ("train", "val"):
    n = len(list(Path(f"{PREPROCESSED_CACHE}/{split}").glob("*.npz")))
    print(f"  {split}/: {n} .npz files")
print(f"  Split CSV: {'OK' if os.path.exists(SPLIT_CSV) else 'MISSING'}")


## 3. Sanity check â€” cache loads, 4 channels, phase aug fires

In [ ]:
import sys
sys.path.insert(0, REPO_DIR)

import numpy as np
from pathlib import Path

train_dir = Path(PREPROCESSED_CACHE) / "train"
val_dir   = Path(PREPROCESSED_CACHE) / "val"

assert train_dir.exists(), f"train/ missing in cache: {train_dir}"
assert val_dir.exists(),   f"val/ missing in cache: {val_dir}"

train_npz = sorted(train_dir.glob("*.npz"))
val_npz   = sorted(val_dir.glob("*.npz"))
print(f"train/: {len(train_npz)} .npz files (expected ~1200)")
print(f"val/  : {len(val_npz)} .npz files (expected ~306)")

# Inspect one to verify channel count
sample = np.load(train_npz[0])
print(f"\nSample {train_npz[0].stem}:")
print(f"  image: {sample['image'].shape}  (expected (4, H, W, D))")
print(f"  label: {sample['label'].shape}  (expected (1, H, W, D))")
print(f"  image dtype: {sample['image'].dtype}")
assert sample["image"].shape[0] == 4, f"Wrong channel count: {sample['image'].shape}"

# Per-channel stats
for c, name in enumerate(["pre", "post1", "post2", "subtraction"]):
    arr = sample["image"][c]
    print(f"  ch{c} ({name}): min={arr.min():.3f}, max={arr.max():.3f}, mean={arr.mean():.3f}")

In [ ]:
# Verify phase aug transform actually permutes channels when applied
from data.mama_mia_dataset_4ch import PhaseAugment4Ch
import torch

img = torch.from_numpy(sample["image"].astype(np.float32))
aug = PhaseAugment4Ch(keys=["image"], p_apply=1.0)  # always apply

out = aug({"image": img.clone()})
perm_ok = not torch.allclose(out["image"][1:], img[1:])
ch0_unchanged = torch.allclose(out["image"][0], img[0])
print(f"Phase aug permuted channels 1-3: {perm_ok}")
print(f"Phase aug left ch0 (pre) untouched: {ch0_unchanged}")
assert ch0_unchanged, "BUG: phase aug shouldn't touch channel 0"

## 4. Train

In [ ]:
import sys

cmd = [
    "python", f"{REPO_DIR}/training/train_centralized_4ch.py",
    "--config",                 f"{REPO_DIR}/configs/config_4ch.yaml",
    "--data_root",              PREPROCESSED_CACHE,   # only used for split_csv
    "--preprocessed_cache_dir", PREPROCESSED_CACHE,
    "--output_dir",             OUTPUT_DIR,
    "--epochs",                 str(EPOCHS),
    "--batch_size",             str(BATCH_SIZE),
    "--lr",                     str(LR),
    "--num_workers",            "2",
    "--patch_size",             str(PATCH_X), str(PATCH_Y), str(PATCH_Z),
    "--warmup_epochs",          "5",
    "--seed",                   str(SEED),
    "--phase_aug",              str(PHASE_AUG),
    "--p_phase_aug",            str(P_PHASE_AUG),
]
print(" \\\n  ".join(cmd))
print()

import subprocess
subprocess.run(cmd, check=True)

## 5. Final report

Re-reads `train_log.json` + `meta.json` and prints the final Dice. **Download these files (and `best_model.pth`) to your laptop BEFORE the session closes** â€” `/kaggle/working` wipes on exit.

In [ ]:
import json

with open(f"{OUTPUT_DIR}/meta.json") as f:
    meta = json.load(f)
with open(f"{OUTPUT_DIR}/train_log.json") as f:
    log = json.load(f)

val_entries = [e for e in log if "val_dice" in e]

print("=" * 60)
print("4-CHANNEL UNET3D â€” FINAL REPORT")
print("=" * 60)
print(f"Seed         : {meta['seed']}")
print(f"Phase aug    : {meta['phase_aug']} (p={meta.get('p_phase_aug', 0.5)})")
print(f"In channels  : {meta['in_channels']}")
print(f"Epochs       : {meta['epochs']}")
print(f"Best Val Dice: {meta['best_dice']:.4f}")
print()
print("Val checkpoints:")
for e in val_entries:
    print(f"  ep{e['epoch']:>3} | Dice: {e['val_dice']:.4f} | "
          f"IoU: {e['val_iou']:.4f} | Sens: {e['val_sensitivity']:.4f}")

print()
print("=== COMPARISON (2ch baseline s=42 = 0.6791) ===")
delta = meta['best_dice'] - 0.6791
if delta >= 0.020:
    verdict = "REAL WIN (Î” â‰¥ +0.020) â€” justifies redoing FADC variants at 4ch"
elif delta >= 0.005:
    verdict = "marginal (+0.005 â‰¤ Î” < +0.020) â€” future-work, not paper-headline"
else:
    verdict = "no win (Î” < +0.005) â€” file as negative ablation"
print(f"Î” = {delta:+.4f}  â†’  {verdict}")

In [ ]:
# Helper to list what's in OUTPUT_DIR so you remember to download:
import os
for f in sorted(os.listdir(OUTPUT_DIR)):
    sz_mb = os.path.getsize(os.path.join(OUTPUT_DIR, f)) / 1e6
    print(f"  {f}  ({sz_mb:.1f} MB)")

## 6. Visualize predictions — ground truth, predicted mask, error overlap

Loads `best_model.pth`, picks one val case per collection (DUKE / ISPY1 / ISPY2 / NACT), runs sliding-window inference, and plots a 4-column grid:

**MRI input (post1)** | **Ground Truth (green)** | **Prediction (cyan)** | **TP / FP / FN overlay**

- Yellow = True Positive (correct tumor pixels)
- Red = False Positive (predicted tumor where none exists)
- Green = False Negative (missed tumor pixels)

Output saved to `OUTPUT_DIR/predictions_4ch.png` — download it with the rest of the outputs before the Kaggle session closes.

In [ ]:
import sys, os
import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from pathlib import Path

sys.path.insert(0, REPO_DIR)
from models.unet_3d import UNet3D
from monai.inferers import sliding_window_inference
from monai.transforms import AsDiscrete

COLLECTIONS = ["DUKE", "ISPY1", "ISPY2", "NACT"]
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- Load checkpoint ---
ckpt_path = Path(OUTPUT_DIR) / "best_model.pth"
assert ckpt_path.exists(), f"Checkpoint not found: {ckpt_path}"
ckpt = torch.load(ckpt_path, map_location=DEVICE)
cfg = ckpt["config"]
best_dice = ckpt["best_dice"]
best_epoch = ckpt["epoch"] + 1
patch_size = tuple(cfg["data"]["patch_size"])
print(f"Checkpoint: epoch {best_epoch} | Val Dice {best_dice:.4f} | patch {patch_size}")

# --- Build model and load weights ---
model = UNet3D(
    in_channels=cfg["model"]["in_channels"],
    out_channels=cfg["model"]["out_channels"],
    base_filters=cfg["model"]["base_filters"],
).to(DEVICE)
model.load_state_dict(ckpt["model"])
model.eval()
print(f"Model: UNet3D | in_channels={cfg['model']['in_channels']} (4ch)")

# --- Pick one val case per collection (first alphabetical for reproducibility) ---
val_dir = Path(PREPROCESSED_CACHE) / "val"
selected = []
for col in COLLECTIONS:
    matches = sorted(val_dir.glob(f"{col.lower()}_*.npz"))
    if matches:
        selected.append({"patient_id": matches[0].stem, "collection": col, "path": matches[0]})
    else:
        print(f"WARNING: no val cases found for collection {col}")
print(f"Selected {len(selected)} cases: {[c['patient_id'] for c in selected]}")

# --- Run inference + compute Dice per case ---
post_pred = AsDiscrete(argmax=True)
results = []
for c in selected:
    data = np.load(c["path"])
    image_np = data["image"].astype(np.float32)
    label_np = data["label"].astype(np.float32)
    assert image_np.shape[0] == 4, f"Expected 4 channels, got {image_np.shape[0]}"

    image_t = torch.from_numpy(image_np).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        logits = sliding_window_inference(
            image_t, patch_size, sw_batch_size=4, predictor=model, overlap=0.25
        )
    pred_cls = post_pred(logits[0]).cpu().numpy()
    pred_fg  = (pred_cls[0] == 1).astype(np.float32)
    label_fg = label_np[0]
    image_post = image_np[1]  # post1 = first contrast-enhanced phase

    tp = (pred_fg * label_fg).sum()
    denom = pred_fg.sum() + label_fg.sum()
    dice = float(2 * tp / (denom + 1e-6))

    sums = label_fg.sum(axis=(0, 1))
    z = int(sums.argmax()) if sums.max() > 0 else image_post.shape[2] // 2

    print(f"  {c['patient_id']:25s}  Dice={dice:.4f}  best_slice z={z}")
    results.append({**c, "image": image_post, "gt": label_fg, "pred": pred_fg, "dice": dice, "z": z})

# --- Plot 4 columns x N rows ---
n = len(results)
fig, axes = plt.subplots(n, 4, figsize=(20, 5 * n))
if n == 1:
    axes = axes[np.newaxis, :]

col_titles = [
    "MRI Input (post1)",
    "Ground Truth (green)",
    "Prediction (cyan)",
    "Error Overlay (TP / FP / FN)",
]

for i, r in enumerate(results):
    img, gt, pred, z = r["image"], r["gt"], r["pred"], r["z"]
    img_norm = (img - img.min()) / (img.max() - img.min() + 1e-8)

    img_s  = img_norm[:, :, z].T
    gt_s   = gt[:, :, z].T
    pred_s = pred[:, :, z].T

    axes[i, 0].imshow(img_s, cmap="gray", origin="lower")
    axes[i, 0].set_ylabel(
        f"{r['patient_id']}\n({r['collection']})\nDice = {r['dice']:.3f}",
        fontsize=10,
    )

    axes[i, 1].imshow(img_s, cmap="gray", origin="lower")
    gt_rgba = np.zeros((*gt_s.shape, 4))
    gt_rgba[gt_s > 0] = [0.0, 1.0, 0.0, 0.55]
    axes[i, 1].imshow(gt_rgba, origin="lower")

    axes[i, 2].imshow(img_s, cmap="gray", origin="lower")
    pred_rgba = np.zeros((*pred_s.shape, 4))
    pred_rgba[pred_s > 0] = [0.0, 0.85, 1.0, 0.55]
    axes[i, 2].imshow(pred_rgba, origin="lower")

    axes[i, 3].imshow(img_s, cmap="gray", origin="lower")
    overlay = np.zeros((*gt_s.shape, 4))
    overlay[(gt_s > 0)  & (pred_s > 0)]  = [1.00, 1.00, 0.00, 0.65]
    overlay[(gt_s == 0) & (pred_s > 0)]  = [1.00, 0.20, 0.20, 0.65]
    overlay[(gt_s > 0)  & (pred_s == 0)] = [0.20, 1.00, 0.20, 0.65]
    axes[i, 3].imshow(overlay, origin="lower")

    for ax in axes[i]:
        ax.set_xticks([])
        ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_visible(False)

for j, title in enumerate(col_titles):
    axes[0, j].set_title(title, fontsize=12, fontweight="bold", pad=8)

legend_handles = [
    Patch(facecolor="yellow", label="True Positive (TP)"),
    Patch(facecolor="red",    label="False Positive (FP)"),
    Patch(facecolor="lime",   label="False Negative (FN)"),
]
fig.legend(handles=legend_handles, loc="lower center", ncol=3,
           fontsize=11, bbox_to_anchor=(0.5, -0.01), framealpha=0.95)

title = f"4ch 3D U-Net (seed={SEED}) -- Best Val Dice: {best_dice:.4f} | Epoch {best_epoch}"
plt.suptitle(title, fontsize=14, fontweight="bold", y=1.005)

plt.tight_layout()
out_path = Path(OUTPUT_DIR) / "predictions_4ch.png"
plt.savefig(out_path, dpi=150, bbox_inches="tight")
plt.show()

mean_dice = np.mean([r["dice"] for r in results])
print(f"\nMean Dice across {len(results)} sampled cases: {mean_dice:.4f}")
print(f"Figure saved -> {out_path}")
